In [10]:
# ◯ Librerías necesarias
# ✓ pyodbc: permite conectar Python con bases de datos SQL Server (ODBC)
# ✓ pandas: para manipulación y análisis de datos

import pyodbc
import pandas as pd

In [11]:
# ◯ Configurar la conexión a SQL Server (ajustar según tu servidor local o remoto)

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=VALESURFACE\\SQLEXPRESS;"            # Cambia esto si usás otro nombre de servidor o una IP
    "DATABASE=Northwind;"                        # Asegurate de que la base esté cargada y activa
    "Trusted_Connection=yes;"                    # Cambiar a UID/PWD si no usás login de Windows
)

print("✅ Conexión exitosa")

OperationalError: ('HYT00', '[HYT00] [Microsoft][ODBC Driver 17 for SQL Server]Login timeout expired (0) (SQLDriverConnect)')

In [ ]:


# ◯ Verificar que la conexión funciona mostrando las tablas disponibles
query_tablas = "SELECT name FROM sys.tables;"
tablas = pd.read_sql(query_tablas, conn)

# ◯ Mostrar primeras filas
print("✅ Conexión exitosa. Tablas disponibles:")
display(tablas)

In [ ]:
# Conectar a la base de datos SQLite
conexion = sqlite3.connect('sanoyfresco.db')

Nota: si la carga de la bbdd da un error de que está corrupta asegurarse de que ha terminado de cargarse, y luego hacer click en los 3 puntitos del nombre la bbdd y en actualizar

In [ ]:
# Cargar una tabla completa en un DataFrame
df = pd.read_sql_query("SELECT * FROM tickets", conexion)

# Mostrar los primeros registros del DataFrame
df

In [ ]:
# Cerrar la conexión a la base de datos
conexion.close()

In [ ]:
df.info()

In [ ]:
df['fecha'] = pd.to_datetime(df['fecha'])

In [ ]:
df_cesta = df[['id_pedido','nombre_producto']]
df_cesta

In [ ]:
# Agrupar los productos por id_pedido
df_agrupado = df_cesta.groupby('id_pedido')['nombre_producto'].apply(lambda producto: ','.join(producto))
df_agrupado

In [ ]:
# Aplicar pd.get_dummies() para transformar los productos en columnas con 0/1
df_transacciones = df_agrupado.str.get_dummies(sep=',')
df_transacciones

In [ ]:
# Soporte para cada producto
soporte = df_transacciones.mean() * 100
soporte.sort_values(ascending=False)

In [ ]:
# Función para calcular la confianza entre dos productos en la muestra
def confianza(antecedente, consecuente):
    # Casos donde se compraron ambos productos
    conjunto_ac = df_transacciones[(df_transacciones[antecedente] == 1) &
                                   (df_transacciones[consecuente] == 1)]
    # Confianza = compras conjuntas / compras de producto A
    return len(conjunto_ac) / df_transacciones[antecedente].sum()


In [ ]:
# Función para calcular el lift entre dos productos en la muestra
def lift(antecedente, consecuente):
    soporte_a = df_transacciones[antecedente].mean()
    soporte_c = df_transacciones[consecuente].mean()
    conteo_ac = len(df_transacciones[(df_transacciones[antecedente] == 1) &
                                   (df_transacciones[consecuente] == 1)])
    soporte_ac = conteo_ac / len(df_transacciones)
    return soporte_ac / (soporte_a * soporte_c)

In [ ]:
from itertools import combinations

# Definir un umbral para la confianza mínima
umbral_confianza = 0.05
asociaciones = []

# Generar combinaciones de productos y calcular confianza y lift
for antecedente, consecuente in combinations(df_transacciones.columns, 2):

    # Soporte del antecedente
    soporte_a = df_transacciones[antecedente].mean()

    # Calcular confianza
    conf = confianza(antecedente, consecuente)
    if conf > umbral_confianza:
        asociaciones.append({
            'antecedente': antecedente,
            'consecuente': consecuente,
            'soporte_a': round(soporte_a * 100,1),
            'confianza': round(conf * 100,1),
            'lift': round(lift(antecedente, consecuente),1)
        })


# Convertir las asociaciones en un DataFrame
df_asociaciones = pd.DataFrame(asociaciones)

# Ordenar las asociaciones por confianza de mayor a menor
df_asociaciones.sort_values(by='lift', ascending=False, inplace=True)

df_asociaciones

Generamos una tabla de los productos únicos con su 'id_producto', 'id_seccion', 'id_departamento' para enriquecer la tabla de reglas

In [ ]:
# Crear una tabla con los productos únicos y las columnas correspondientes
productos_unicos = df[['id_producto', 'id_seccion', 'id_departamento', 'nombre_producto']].drop_duplicates()

productos_unicos

In [ ]:
df_asociaciones_enriquecido = df_asociaciones.merge(productos_unicos, left_on='antecedente', right_on='nombre_producto', how='left').drop(columns=['nombre_producto'])
df_asociaciones_enriquecido.columns = ['antecedente', 'consecuente', 'soporte_a', 'confianza', 'lift', 'id_producto_a', 'id_seccion_a', 'id_departamento_a']
df_asociaciones_enriquecido

In [ ]:
df_asociaciones_enriquecido.to_csv('reglas.csv', index=False, sep=';', decimal=',')
